<a href="https://colab.research.google.com/github/srihari-g-acharya-ci109/ML2LAB/blob/main/ML8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# load iris dataset
iris = load_iris()
X = iris.data
y = iris.target
features = iris.feature_names

df = pd.DataFrame(X, columns=features)
df["true_species"] = y

print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
# standard scaling is essential because k-means relies on euclidean distance
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# elbow method and silhouette score to verify optimal k
k_values = range(2, 9)
wcss = []
sil_scores = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    wcss.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(list(k_values), wcss, marker='o', color='purple')
axes[0].set_title("Elbow Method (WCSS vs k)")
axes[0].set_xlabel("Number of Clusters (k)")
axes[0].set_ylabel("Inertia")
axes[0].grid(True, linestyle="--", alpha=0.5)

axes[1].plot(list(k_values), sil_scores, marker='s', color='darkorange')
axes[1].set_title("Silhouette Score vs k")
axes[1].set_xlabel("Number of Clusters (k)")
axes[1].set_ylabel("Silhouette Score")
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

print("Best k by silhouette score:", list(k_values)[np.argmax(sil_scores)])

In [ ]:
# fit final model with k=3
k_optimal = 3
kmeans = KMeans(n_clusters=k_optimal, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)

print(f"Silhouette Score (k=3): {silhouette_score(X_scaled, df['cluster']):.4f}")
print(f"Inertia: {kmeans.inertia_:.2f}")

# centroids back to original units
centroids_orig = scaler.inverse_transform(kmeans.cluster_centers_)
centroid_df = pd.DataFrame(centroids_orig, columns=features)
centroid_df.index = [f"Cluster {i}" for i in range(k_optimal)]
print("
Cluster Centroids (Original Scale):")
display(centroid_df)

# cross-tabulate with true species
print("
Cluster vs True Species alignment:")
print(pd.crosstab(df["true_species"], df["cluster"], rownames=["True Species"], colnames=["KMeans Cluster"]))

In [ ]:
# 2D scatter plot: Sepal Length vs Petal Length
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df,
    x="sepal length (cm)",
    y="petal length (cm)",
    hue="cluster",
    palette="Set2",
    s=70,
    edgecolor="black"
)

# overlay cluster centers
plt.scatter(
    centroids_orig[:, 0],
    centroids_orig[:, 2],
    marker="X",
    s=250,
    c="red",
    edgecolor="black",
    linewidth=1.5,
    label="Centroids"
)

plt.title("K-Means Clustering on Iris (Sepal Length vs Petal Length)")
plt.xlabel("Sepal Length (cm)")
plt.ylabel("Petal Length (cm)")
plt.legend()
plt.tight_layout()
plt.show()